# Phase 3: Data Preparation

**CRISP-DM Phase Description:**  
This phase covers all activities to construct the final dataset from the initial raw data. Data preparation tasks are likely to be performed multiple times, and not in any prescribed order. This is typically the longest and most time-consuming phase of the CRISP-DM lifecycle.

---

In [2]:
# Standard library imports for this phase
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
%matplotlib inline

In [4]:
# Load the dataset from Phase 2 (update the path as needed)
df = pd.read_csv("data/california_housing_phase2.csv")
df.head()

print(f"Loaded dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

Loaded dataset: 20640 rows x 9 columns


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


---
### Task 1: Select Data

Decide on the data to be used for analysis. Consider which columns (features) and rows (records) to include or exclude based on:

- **Relevance:** Does this feature contribute to the data mining goal?
- **Data Quality:** Is the quality of this feature sufficient (e.g., too many missing values)?
- **Technical Constraints:** Are there limitations on data volume or specific feature types?

**Output:** A rationale for inclusion/exclusion of data, and the resulting subset.

**Instructions:** Select the columns and rows relevant to your analysis goal. Document your reasoning.

In [4]:
# Select the relevant columns and rows for the analysis.
# Rationale:
# - Keep all available features because each one may contribute to house value prediction.
# - Keep the target column 'MedHouseVal' because it is the variable to predict.
# - No columns are excluded because the California Housing dataset is already compact,
#   numeric, and relevant to the project goal.

columns_to_keep = df.columns.tolist()
columns_to_drop = []

selection_rationale = {
    'MedInc': 'Strong economic indicator and usually highly predictive of house value.',
    'HouseAge': 'Useful property characteristic that may influence value.',
    'AveRooms': 'Represents housing size and living space.',
    'AveBedrms': 'Adds more detail about home structure.',
    'Population': 'Captures neighborhood density and local demand context.',
    'AveOccup': 'Provides information about occupancy patterns.',
    'Latitude': 'Important location feature.',
    'Longitude': 'Important location feature.',
    'MedHouseVal': 'Target variable for prediction.'
}

df_selected = df[columns_to_keep].copy()

print(f"Shape after column selection: {df_selected.shape}")
print("Columns kept:", columns_to_keep)
print("\nSelection rationale:")
for col, reason in selection_rationale.items():
    print(f"- {col}: {reason}")

Shape after column selection: (20640, 9)
Columns kept: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'MedHouseVal']

Selection rationale:
- MedInc: Strong economic indicator and usually highly predictive of house value.
- HouseAge: Useful property characteristic that may influence value.
- AveRooms: Represents housing size and living space.
- AveBedrms: Adds more detail about home structure.
- Population: Captures neighborhood density and local demand context.
- AveOccup: Provides information about occupancy patterns.
- Latitude: Important location feature.
- Longitude: Important location feature.
- MedHouseVal: Target variable for prediction.


In [ ]:
# Optional: Filter rows based on specific criteria
# Example: Remove rows where a critical field is missing or filter by a condition

df_selected = df_selected.copy()
print(f"Shape after row selection: {df_selected.shape}")

Shape after row selection: (20640, 9)


---
### Task 2: Clean Data

Raise data quality to the level required by the selected analysis techniques. Cleaning activities include:

- **Handle Missing Values:** Impute missing values (mean, median, mode, forward/backward fill) or remove rows/columns with excessive missing data.
- **Correct Errors:** Fix inaccurate or corrupted data entries.
- **Remove Duplicates:** Eliminate exact or near-duplicate records.
- **Handle Outliers:** Decide how to treat extreme values (keep, cap, transform, or remove).

**Instructions:** Apply appropriate cleaning techniques to address the data quality issues identified in Phase 2, Task 4.

In [9]:
# Handle missing values.
# The California Housing dataset contains no missing values, so no imputation is required.
df_clean = df_selected.copy()

missing_values = df_clean.isnull().sum()
print("Missing values per column:")
print(missing_values)

print("\nTotal missing values:", missing_values.sum())
print("No missing-value treatment was required.")

Missing values per column:
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Total missing values: 0
No missing-value treatment was required.


In [10]:
# Remove duplicate records.

before = len(df_clean)
duplicate_count = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
after = len(df_clean)

print(f"Duplicate rows found: {duplicate_count}")
print(f"Removed {before - after} duplicate rows. Remaining: {after} rows.")

Duplicate rows found: 0
Removed 0 duplicate rows. Remaining: 20640 rows.


In [11]:
# Handle outliers.
# Strategy used: cap extreme values in numerical feature columns using the IQR method.
# The target column is not capped so the real house values are preserved.

def cap_outliers_iqr(dataframe, column):
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    dataframe[column] = dataframe[column].clip(lower=lower_bound, upper=upper_bound)
    return lower_bound, upper_bound

feature_cols = [col for col in df_clean.columns if col != 'MedHouseVal']
outlier_summary = {}

for col in feature_cols:
    lower, upper = cap_outliers_iqr(df_clean, col)
    outlier_summary[col] = {'lower_bound': lower, 'upper_bound': upper}

print("Outlier capping completed for feature columns.")
pd.DataFrame(outlier_summary).T

Outlier capping completed for feature columns.


,lower_bound,upper_bound
MedInc,-0.706375,8.013025
HouseAge,-10.500000,65.500000
AveRooms,2.023219,8.469878
AveBedrms,0.865909,1.239697
Population,-620.000000,3132.000000
AveOccup,1.150961,4.561041
Latitude,28.260000,43.380000
Longitude,-127.485000,-112.325000


---
### Task 3: Construct Data (Feature Engineering)

This task involves creating new attributes (features) derived from existing ones that may be more useful for modelling. Common techniques include:

- **Derived Attributes:** Create new features from existing ones (e.g., extracting `year`, `month`, `day` from a datetime column; computing `total_spend = price * quantity`).
- **Binning / Discretisation:** Convert continuous variables into categorical bins (e.g., age groups).
- **Encoding Categorical Variables:** Convert categorical features into numerical representations (e.g., one-hot encoding, label encoding).
- **Scaling / Normalisation:** Scale numerical features to a common range (e.g., Min-Max scaling, Standardisation).

**Instructions:** Create new features or transform existing ones to improve model performance.

In [13]:
# Create derived attributes / new features.

# Feature 1: Rooms per household proxy
df_clean['RoomsPerOccupant'] = df_clean['AveRooms'] / df_clean['AveOccup']

# Feature 2: Bedrooms ratio
df_clean['BedroomRoomRatio'] = df_clean['AveBedrms'] / df_clean['AveRooms']

# Feature 3: Population per room
df_clean['PopulationPerRoom'] = df_clean['Population'] / df_clean['AveRooms']

print("New features created:")
print(['RoomsPerOccupant', 'BedroomRoomRatio', 'PopulationPerRoom'])
df_clean.head()

New features created:
['RoomsPerOccupant', 'BedroomRoomRatio', 'PopulationPerRoom']


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,RoomsPerOccupant,BedroomRoomRatio,PopulationPerRoom
0,8.013025,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526,2.732919,0.146591,46.104545
1,8.013025,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585,2.956685,0.155797,384.890548
2,7.257400,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521,2.957661,0.129516,59.844581
3,5.643100,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413,2.283154,0.184458,95.919937
4,3.846200,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422,2.879646,0.172096,89.941610


In [14]:
# Encode categorical variables.
# No categorical variables are present in the California Housing dataset,
# so no encoding is required.

print("No categorical variables found. Encoding step not required.")

No categorical variables found. Encoding step not required.


In [15]:
# Scale / normalise numerical features if required.
from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler = StandardScaler()

target_col = 'MedHouseVal'
feature_cols = [col for col in df_clean.columns if col != target_col]

df_scaled = df_clean.copy()
df_scaled[feature_cols] = scaler.fit_transform(df_scaled[feature_cols])

print("Scaled feature columns:")
print(feature_cols)
df_scaled.head()

Scaled feature columns:
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'RoomsPerOccupant', 'BedroomRoomRatio', 'PopulationPerRoom']


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,RoomsPerOccupant,BedroomRoomRatio,PopulationPerRoom
0,2.541006,0.982143,1.347665,-0.424488,-1.325821,-0.497871,1.052548,-1.327835,4.526,1.337638,-1.154776,-1.227312
1,2.541006,-0.607019,0.749027,-1.070004,1.389936,-1.142781,1.043185,-1.322844,3.585,1.709811,-0.989764,0.622212
2,2.085156,1.856182,2.394098,0.192534,-1.098528,-0.140910,1.038503,-1.332827,3.521,1.711435,-1.460845,-1.152302
3,1.111288,1.856182,0.411358,0.187723,-1.017539,-0.508882,1.038503,-1.337818,3.413,0.589576,-0.475999,-0.955357
4,0.027262,1.856182,0.784108,0.287439,-1.008395,-1.039145,1.038503,-1.337818,3.422,1.581678,-0.697598,-0.987994


---
### Task 4: Integrate Data

If your project uses multiple data sources, this task involves merging or combining them into a single, unified dataset. Activities include:

- **Merging Tables:** Join datasets on common keys (e.g., using `pd.merge()`).
- **Appending Records:** Concatenate datasets with the same structure (e.g., using `pd.concat()`).
- **Aggregation:** Summarise data at a different level of granularity.

**Instructions:** If using multiple data sources, merge or concatenate them below. If your project uses a single dataset, document that here and proceed to the next task.

In [16]:
# Integrate data from multiple sources (if applicable).
# This project uses a single data source, so no merging is required.

df_integrated = df_scaled.copy()
print(f"Integrated dataset shape: {df_integrated.shape}")

Integrated dataset shape: (20640, 12)


In [17]:
# Optional: Verify the integrated data
df_integrated.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal,RoomsPerOccupant,BedroomRoomRatio,PopulationPerRoom
0,2.541006,0.982143,1.347665,-0.424488,-1.325821,-0.497871,1.052548,-1.327835,4.526,1.337638,-1.154776,-1.227312
1,2.541006,-0.607019,0.749027,-1.070004,1.389936,-1.142781,1.043185,-1.322844,3.585,1.709811,-0.989764,0.622212
2,2.085156,1.856182,2.394098,0.192534,-1.098528,-0.140910,1.038503,-1.332827,3.521,1.711435,-1.460845,-1.152302
3,1.111288,1.856182,0.411358,0.187723,-1.017539,-0.508882,1.038503,-1.337818,3.413,0.589576,-0.475999,-0.955357
4,0.027262,1.856182,0.784108,0.287439,-1.008395,-1.039145,1.038503,-1.337818,3.422,1.581678,-0.697598,-0.987994


---
### Task 5: Format Data

This final preparation task ensures the data is in the correct format for the modelling tools. Activities include:

- **Data Type Conversions:** Ensure all columns have appropriate data types (e.g., numeric, datetime, categorical).
- **Column Reordering:** Arrange columns in a logical order (e.g., features first, target last).
- **Renaming:** Give columns clear, descriptive names.
- **Saving the Prepared Dataset:** Export the final, clean dataset for use in subsequent phases.

**Instructions:** Apply any final formatting changes and save the prepared dataset.

In [ ]:
# Apply final formatting — data types, column order, renaming.

# Rename columns for clarity
df_integrated = df_integrated.rename(columns={
    'MedInc': 'MedianIncome',
    'HouseAge': 'HouseAge',
    'AveRooms': 'AverageRooms',
    'AveBedrms': 'AverageBedrooms',
    'Population': 'Population',
    'AveOccup': 'AverageOccupancy',
    'Latitude': 'Latitude',
    'Longitude': 'Longitude',
    'MedHouseVal': 'MedianHouseValue',
    'RoomsPerOccupant': 'RoomsPerOccupant',
    'BedroomRoomRatio': 'BedroomRoomRatio',
    'PopulationPerRoom': 'PopulationPerRoom'
})

# Reorder columns (features first, target last)
target_col = 'MedianHouseValue'
feature_cols = [col for col in df_integrated.columns if col != target_col]
df_final = df_integrated[feature_cols + [target_col]].copy()

print("Column order after formatting:")
print(df_final.columns.tolist())
print("\nData types:")
print(df_final.dtypes)

Column order after formatting:
['MedianIncome', 'HouseAge', 'AverageRooms', 'AverageBedrooms', 'Population', 'AverageOccupancy', 'Latitude', 'Longitude', 'RoomsPerOccupant', 'BedroomRoomRatio', 'PopulationPerRoom', 'MedianHouseValue']

Data types:
MedianIncome         float64
HouseAge             float64
AverageRooms         float64
AverageBedrooms      float64
Population           float64
AverageOccupancy     float64
Latitude             float64
Longitude            float64
RoomsPerOccupant     float64
BedroomRoomRatio     float64
PopulationPerRoom    float64
MedianHouseValue     float64
dtype: object


In [19]:
# Verify the final prepared dataset.

print("=" * 60)
print("FINAL PREPARED DATASET SUMMARY")
print("=" * 60)
print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final.isnull().sum().sum()}")
print(f"Duplicates: {df_final.duplicated().sum()}")
print("\nColumn types:")
print(df_final.dtypes)
print("\nPreview:")
df_final.head()

FINAL PREPARED DATASET SUMMARY
Shape: (20640, 12)
Missing values: 0
Duplicates: 0

Column types:
MedianIncome         float64
HouseAge             float64
AverageRooms         float64
AverageBedrooms      float64
Population           float64
AverageOccupancy     float64
Latitude             float64
Longitude            float64
RoomsPerOccupant     float64
BedroomRoomRatio     float64
PopulationPerRoom    float64
MedianHouseValue     float64
dtype: object

Preview:


,MedianIncome,HouseAge,AverageRooms,AverageBedrooms,Population,AverageOccupancy,Latitude,Longitude,RoomsPerOccupant,BedroomRoomRatio,PopulationPerRoom,MedianHouseValue
0,2.541006,0.982143,1.347665,-0.424488,-1.325821,-0.497871,1.052548,-1.327835,1.337638,-1.154776,-1.227312,4.526
1,2.541006,-0.607019,0.749027,-1.070004,1.389936,-1.142781,1.043185,-1.322844,1.709811,-0.989764,0.622212,3.585
2,2.085156,1.856182,2.394098,0.192534,-1.098528,-0.140910,1.038503,-1.332827,1.711435,-1.460845,-1.152302,3.521
3,1.111288,1.856182,0.411358,0.187723,-1.017539,-0.508882,1.038503,-1.337818,0.589576,-0.475999,-0.955357,3.413
4,0.027262,1.856182,0.784108,0.287439,-1.008395,-1.039145,1.038503,-1.337818,1.581678,-0.697598,-0.987994,3.422


In [21]:
# Save the prepared dataset for use in Phase 4 (Modelling).

OUTPUT_PATH = 'california_housing_phase3.csv'
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Prepared dataset saved to: {OUTPUT_PATH}")

Prepared dataset saved to: california_housing_phase3.csv
